# W06a 실습 A · MF의 분리와 검증

**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2**

2026년 10월 6일 · W06a · 주교재 4.4–4.6.

CPU 런타임에서 첫 셀부터 실행합니다. 파일 → Drive에 사본 저장으로 자신의 실습본을 만드세요.
설치 후 재시작 안내가 나오면 런타임을 재시작하고 첫 셀부터 다시 실행합니다.
개인 PC 경로나 파일 업로드 없이 공식 MovieLens를 자동 준비합니다. 네트워크 실패는 오류로 알립니다.
빈칸은 문제용 변수로 분리했습니다. 미완성 문제를 건너뛰어도 독립 실험은 실행됩니다.
자동 검사 통과가 자신의 모든 설명·빈칸이 정답임을 뜻하지는 않습니다.

목표: 관측과 미관측, 분할, 평균 계산, RMSE, 선택 epoch를 숫자로 확인합니다.


## 실행 환경 준비

공통 구현을 가져옵니다. 코드를 복사해 서로 다른 MF를 만드는 대신 같은 함수를 사용합니다.


In [ ]:
import os, sys, subprocess
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w06a'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
from luna_recsys.mf_sgd import MFSGD,sgd_step
from luna_recsys.model_comparison import comparison_toy,definition_link
from luna_recsys.datasets import prepare_movielens
from luna_recsys.mf_evaluation import (ThreeWaySplit,ValidatedMF,split_three_way,
    predict_rows,rating_metrics,evaluate_mf,fit_validated,search_mf,svd_reconstruct)
from luna_recsys.mf_evaluation_lab import (history_figure,validation_view,
    comparison_view,svd_view,build_evaluation_app)
print('Python',sys.version.split()[0],'고정 수업 버전',COURSE_VERSION)


## 정의 바로가기와 입력·출력

아래 링크는 설치한 고정 버전의 정의 행으로 연결됩니다. 함수 이름을 누르고 docstring부터 읽으세요.

| 함수 | 입력 | 출력과 주의점 |
|---|---|---|
| split_three_way | user_id/movie_id/rating 관측표 | train/validation/test 사본; 중복 쌍 거부 |
| fit_validated | train, validation, K·α·β·epochs | ValidatedMF; 최저 검증 상태 복원, 전체 history 유지 |
| search_mf | train, validation, 후보 | 선택 모델과 비교표; test 인수 없음 |
| evaluate_mf | 학습된 MFSGD, 관측표 | n/RMSE/MAE/새 ID 대체 건수; clipping 없음 |
| svd_reconstruct | 완전한 2D 행렬, rank | 인수·재구성·오차; NaN을 자동 대입하지 않음 |
| build_evaluation_app | train, validation, report_fn | Gradio Blocks; launch는 따로 실행 |

fit_validated는 기존 MFSGD의 초기화와 sgd_step을 재사용합니다. history의 마지막 행과 복원된 model의 상태는 다를 수 있습니다.


In [ ]:
apis=[MFSGD,MFSGD.fit,MFSGD.predict,MFSGD.explain,MFSGD.rmse,sgd_step,
      ThreeWaySplit,ValidatedMF,split_three_way,predict_rows,rating_metrics,
      evaluate_mf,fit_validated,search_mf,svd_reconstruct,prepare_movielens,
      comparison_toy,definition_link,history_figure,validation_view,
      comparison_view,svd_view,build_evaluation_app]
display(Markdown('\n'.join(f'- [{f.__qualname__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


## 1. 작은 관측표 나누기

5×5의 18개 관측을 분리합니다. 원래 물음표였던 칸을 0점으로 만들지 않습니다. 아래 test 표는 절차를 이해하는 작은 합성 예제이며, 실제 데이터의 test 평점은 뒤에서 선택이 끝난 후 사용합니다.

![관측과 분할](../../course/notion/assets/w06a/split-mask.png)


In [ ]:
toy=comparison_toy().ratings.copy()
small=split_three_way(toy)
for name,rows in [('train',small.train),('validation',small.validation),('test',small.test)]:
    print(name,len(rows));display(rows)
pairs=lambda rows:set(rows[['user_id','movie_id']].itertuples(index=False,name=None))


In [ ]:
assert len(toy)==18
assert not (pairs(small.train)&pairs(small.validation) or pairs(small.train)&pairs(small.test) or pairs(small.validation)&pairs(small.test))
assert pairs(small.train)|pairs(small.validation)|pairs(small.test)==pairs(toy)


### A1 · 빈칸: 분리해야 하는 것은 무엇인가요?

학습과 검증의 사용자 ID가 겹쳐도 되나요? 중복되면 안 되는 관측쌍의 교집합을 구하는 식을 작성하세요. 힌트: pairs(표)는 (사용자,영화) 집합입니다.


In [ ]:
a1_overlap=None
a1_reason=""


## 2. 오차를 직접 모으기

실제값−예측값, 절댓값, 제곱을 순서대로 계산합니다. RMSE에는 정규화 항을 더하지 않습니다.


In [ ]:
actual=np.array([5.,3.,1.,4.]);prediction=np.array([4.,3.,2.,3.])
errors=actual-prediction
display(pd.DataFrame({'actual':actual,'prediction':prediction,'error':errors,'absolute':abs(errors),'squared':errors**2}))
print(rating_metrics(actual,prediction))


In [ ]:
assert np.isclose(rating_metrics(actual,prediction)['rmse'],np.sqrt(.75))
assert rating_metrics(actual,prediction)['mae']==.75


### A2 · 디버깅: 평균 계산의 범위

잘못된 예시 `mu = ratings.rating.mean()`을 train/validation/test 분리 후 그대로 사용했습니다. 어느 표로 계산해야 하는지 수정하고 이유를 쓰세요.


In [ ]:
a2_fixed_code=""
a2_reason=""


## 3. 실제 MovieLens의 고정 빠른 모드

공식 데이터를 다운로드하거나 캐시를 재사용합니다. 20K 표본의 드문 사용자 때문에 층화가 불가능하면 무작위 분할로 대체하고 실제 방법을 출력합니다. `random-small-data`에는 드문 사용자로 인한 대체도 포함됩니다.


In [ ]:
prepared=prepare_movielens('data/local/w06a',mode='real')
assert prepared.mode in {'movielens','local'}
print(prepared.description)
all_ratings=prepared.data.ratings
assert len(all_ratings)==100000
ratings=all_ratings.sample(n=20000,random_state=20261006).reset_index(drop=True)
split=split_three_way(ratings,seed=20261006)
print('빠른 모드: 20,000개 관측 표본, 전체 100K 결과가 아닙니다.')
print('train / validation / test:',len(split.train),len(split.validation),len(split.test))
print('실제 분할 방법:',split.methods)


In [ ]:
assert (len(split.train),len(split.validation),len(split.test))==(12000,4000,4000)
assert not pairs(split.train)&pairs(split.test)
assert not pairs(split.train)&pairs(split.validation)


## 4. 한 후보의 학습·검증 곡선

K=2, α=0.02, β=0.2, 최대 20 epoch를 실행합니다. 선택을 설명하는 예제이며 아직 test를 채점하지 않습니다. 최저 검증 상태를 복원하므로 마지막 history 행이 현재 모델의 상태와 같다고 가정하지 마세요.


In [ ]:
result=fit_validated(split.train,split.validation,n_factors=2,regularization=.2,epochs=20)
display(result.history)
display(history_figure(result.history,result.best_epoch))
print('선택 epoch',result.best_epoch,'검증 RMSE',result.best_validation_rmse,'실행 초',round(result.seconds,2))
print('학습 평균',result.model.mean_)


In [ ]:
assert len(result.history)==21
assert np.isclose(result.model.mean_,split.train.rating.mean())
assert np.isclose(evaluate_mf(result.model,split.validation)['rmse'],result.history.validation_rmse.min())
assert set(result.model.user_ids_)==set(split.train.user_id)
assert set(result.model.item_ids_)==set(split.train.movie_id)
print('W06A_A_RUNTIME_OK')


### A3 · 비교: 훈련 곡선과 검증 곡선

학습 RMSE가 계속 낮아질 때도 검증 최저 epoch를 고르는 이유를 쓰세요. 복원해야 할 학습 상태 네 가지도 적으세요.


In [ ]:
a3_reason=""
a3_state_names=[]


## 5. 처음 보는 ID를 만났을 때

모델은 그 사용자 취향을 학습하지 않았습니다. 알려진 쪽 편향을 유지한 대체 예측을 확인합니다.


In [ ]:
known_u=result.model.user_ids_[0];known_i=result.model.item_ids_[0]
unknown_u=max(all_ratings.user_id)+1;unknown_i=max(all_ratings.movie_id)+1
for u,i in [(known_u,known_i),(unknown_u,known_i),(known_u,unknown_i),(unknown_u,unknown_i)]:
    info=result.model.explain(u,i)
    print(u,i,{k:v for k,v in info.items() if k!='contributions'})


In [ ]:
assert np.isclose(result.model.predict(unknown_u,unknown_i),result.model.mean_)
assert not result.model.explain(unknown_u,known_i)['known_user']


## 점검 퀴즈

1. 원래 미관측인 칸을 정확도 채점에 쓸 수 있나요?
2. 같은 사용자 ID가 train과 test에 있으면 항상 누출인가요?
3. 학습 평균은 어느 집합에서 구하나요?
4. RMSE와 학습 목적함수는 같나요?
5. 선택 epoch와 마지막 epoch가 다를 때 무엇을 복원하나요?


## 기억할 점과 참고자료

학습은 모수를 바꾸고, 검증은 설정을 고르며, 테스트는 선택이 끝난 모델을 채점합니다.
최적 설정과 지표에는 데이터·분할·후보·대체 규칙을 함께 적으세요.

- 임일(2025), 주교재 4.4–4.6, 79–94쪽.
- [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/)
- [평가와 교차검증](https://scikit-learn.org/stable/modules/cross_validation.html)
- [NumPy SVD](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)
- [Surprise SVD](https://surprise.readthedocs.io/en/latest/matrix_factorization.html)

전체 100K 실행이나 여러 seed 실험은 확장입니다. 새 프로토콜을 정했다면 이전 테스트 결과를 보며 계속 최적화하지 말고 별도의 평가 계획을 먼저 정하세요.

다음은 독립 실행 가능한 실습 B입니다. 네 후보를 비교하고 최종 평가를 마친 뒤 웹 앱 출력 코드를 직접 수정합니다.
